## script to wrangle Xenium data: parse transcripts.parquet file to generate
- temporary a mutationVector format transcript datafile 


In [2]:
import os
import pandas as pd

In [3]:
xenium_bundle_dir = "/mnt/efsPreview/Xenium/hPancreas_Cancer/download/"
transcriptfile = "transcripts.parquet"
transcriptfile = os.path.join(xenium_bundle_dir, transcriptfile)

outputdir = "/mnt/efsPreviewSingleCell/xena/files/xenium/xenium_hPancreas_Cancer"
output_path_transcript_file = "transcripts.tsv"
output_path_transcript_file = os.path.join(outputdir, transcriptfile)

In [25]:
transcript = pd.read_parquet(transcriptfile)
transcript

,transcript_id,cell_id,overlaps_nucleus,feature_name,x_location,y_location,z_location,qv,fov_name,nucleus_distance
0,281474976712210,UNASSIGNED,0,RHOA,207.742142,56.770874,16.747215,40.000000,C1,558.807007
1,281474976712211,UNASSIGNED,0,PFDN5,219.718842,195.691437,16.785574,40.000000,C1,429.642792
2,281474976712949,UNASSIGNED,0,CAPN8,131.823303,114.266029,16.918720,9.477817,C1,545.417603
3,281474976712950,UNASSIGNED,0,HNRNPA2B1,185.292892,138.225296,16.838821,12.165135,C1,496.628540
4,281474976712951,UNASSIGNED,0,ASAH1,191.085968,180.807709,16.914900,40.000000,C1,457.272308
...,...,...,...,...,...,...,...,...,...,...
28455654,281642480447255,UNASSIGNED,0,RHOA,10250.725586,2060.826416,14.072791,16.873135,E17,410.614227
28455655,281642480447256,UNASSIGNED,0,RHOA,10254.545898,2149.297119,14.030017,16.911253,E17,439.560608
28455656,281642480452705,UNASSIGNED,0,RHOA,10251.842773,2043.642212,14.100520,40.000000,E17,408.831482
28455657,281642480452901,UNASSIGNED,0,ATP5MD,10251.536133,2137.854736,14.105377,8.752323,E17,432.560120


In [26]:
transcript = transcript[~(transcript.feature_name.str.startswith("NegControlCodeword"))]
transcript = transcript[~(transcript.feature_name.str.startswith("UnassignedCodeword"))]
transcript = transcript[~(transcript.feature_name.str.startswith("NegControlProbe"))]

In [27]:
transcript.shape, len(set(transcript.feature_name))

((28429402, 10), 477)

In [28]:
transcript = transcript.sort_values(by=['feature_name'])

In [29]:
transcript

,transcript_id,cell_id,overlaps_nucleus,feature_name,x_location,y_location,z_location,qv,fov_name,nucleus_distance
19740449,281487862161203,lemnbooc-1,0,ABCC11,6882.209961,556.627319,18.658958,40.000000,C12,0.093998
12144414,281530811431078,aofhemjg-1,1,ABCC11,4934.902832,208.408691,19.172773,26.982685,C9,0.000000
7634919,281663955427013,eildjfmm-1,1,ABCC11,3087.541260,2002.784180,21.244059,26.982685,E6,0.000000
4602763,281582351053870,mbleekcj-1,1,ABCC11,2064.892578,1125.764893,19.856932,40.000000,D4,0.000000
69954,281608120724294,ehkfieol-1,0,ABCC11,466.949066,2090.109863,19.916914,17.912331,E1,7.536034
...,...,...,...,...,...,...,...,...,...,...
14269651,281603826184292,dglpobgm-1,0,YAF2,5382.446289,977.584106,20.313404,40.000000,D9,1.355766
22314151,281492156597829,ceonhjeg-1,0,YAF2,7603.960938,696.361816,18.307783,40.000000,C13,6.683991
22890220,281625301254513,fedepnbl-1,1,YAF2,7599.963867,1951.489258,19.858097,25.320599,E13,0.000000
6613620,281586646268821,cnnkbolg-1,0,YAF2,2798.333496,1213.806274,20.861923,40.000000,D5,1.130414


### temporary: build transcript table data into mutation vector data 

In [30]:
df = transcript[["feature_name", "x_location", "y_location"]]
df.columns = ["gene", "x", "y"]
df.insert(0, 'reference', [""]* transcript.shape[0])
df.insert(0, 'alt', [""]* transcript.shape[0])
df.insert(0, 'end', [-1]* transcript.shape[0])
df.insert(0, 'start', [-1]* transcript.shape[0])
df.insert(0, 'chr', [""]* transcript.shape[0])
df

,chr,start,end,alt,reference,gene,x,y
19740449,,-1,-1,,,ABCC11,6882.209961,556.627319
12144414,,-1,-1,,,ABCC11,4934.902832,208.408691
7634919,,-1,-1,,,ABCC11,3087.541260,2002.784180
4602763,,-1,-1,,,ABCC11,2064.892578,1125.764893
69954,,-1,-1,,,ABCC11,466.949066,2090.109863
...,...,...,...,...,...,...,...,...
14269651,,-1,-1,,,YAF2,5382.446289,977.584106
22314151,,-1,-1,,,YAF2,7603.960938,696.361816
22890220,,-1,-1,,,YAF2,7599.963867,1951.489258
6613620,,-1,-1,,,YAF2,2798.333496,1213.806274


In [18]:
df.to_csv(output_path_transcript_file, sep="\t")

In [19]:
output_path_transcript_file

'/mnt/efsPreview/Xenium/hPancreas_Cancer/download/transcripts.parquet'